# Lateral profile validation workflow

Reproducible notebook for comparing measured and Monte Carlo lateral profiles for FLASH 6/9 MeV and 10/5/2 cm applicators. It generates processed data, all 18 comparison graphs, pointwise percent-difference plots, 2%/2 mm gamma-index plots/pass rates, FWHM/field-width, 80--20% penumbra, symmetry, Monte Carlo uncertainty, output factors, summary CSVs, publication tables, main-paper figures, and supplementary figures.

In [ ]:
# If needed, install runtime dependencies in your Jupyter environment:
# %pip install numpy pandas matplotlib scipy openpyxl nbconvert

In [ ]:
from pathlib import Path
import re, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.interpolate import PchipInterpolator

ROOT = Path.cwd()
OUT = ROOT / "lateral_profile_validation_outputs"
FIG = OUT / "figures"
TABLE = OUT / "tables"
DATA = OUT / "processed_data"
for p in (OUT, FIG, TABLE, DATA):
    p.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.dpi": 140, "savefig.dpi": 300, "font.size": 10, "axes.labelsize": 10, "axes.titlesize": 11, "legend.fontsize": 8})

ENERGIES = ["flash9", "flash6"]
APPLICATORS_CM = [10, 5, 2]
DEPTHS = {"dmax": "Dmax", "r50": "R50", "mid": "Mid-depth"}
EXCEL_FILE = ROOT / "simsim2.xlsx"
EXCEL_SHEETS = {"flash9": "flash9", "flash6": "flash6"}
GAMMA_PERCENT = 2.0
GAMMA_DTA_MM = 2.0
LOW_DOSE_THRESHOLD_PCT = 10.0


In [ ]:
# Case-specific x-axis controls. Update here if you want to reproduce hand tuning from prfiles2.ipynb exactly.
DEFAULT_TUNE = {"flip": False, "x_scale": 1.0, "x_shift_mm": 0.0, "left_start_mm": -1e9, "left_shift_max_mm": 0.0, "right_start_mm": 1e9, "right_shift_max_mm": 0.0, "blend_width_mm": 25.0, "power": 1.4}
TUNING = {}

def tune_for(energy, app_cm, depth):
    d = DEFAULT_TUNE.copy()
    d.update(TUNING.get((energy, app_cm, depth), {}))
    return d

def smoothstep(z):
    z = np.clip(z, 0, 1)
    return z*z*(3 - 2*z)

def apply_x_tuning(x, energy, app_cm, depth):
    t = tune_for(energy, app_cm, depth)
    x = np.asarray(x, dtype=float).copy()
    if t.get("flip", False):
        x = -x
    x = x * t["x_scale"] + t["x_shift_mm"]
    left_w = smoothstep((t["left_start_mm"] - x) / max(t["blend_width_mm"], 1e-6)) ** t["power"]
    right_w = smoothstep((x - t["right_start_mm"]) / max(t["blend_width_mm"], 1e-6)) ** t["power"]
    return x + left_w*t["left_shift_max_mm"] + right_w*t["right_shift_max_mm"]


In [ ]:
def read_numeric_table(path):
    df = pd.read_csv(path, sep=r"[\s,;]+", engine="python", comment="#", header=None).dropna(axis=1, how="all")
    return df.apply(pd.to_numeric, errors="coerce").dropna(how="all")

def read_sim_profile(path):
    df = read_numeric_table(path)
    if df.shape[1] < 2:
        raise ValueError(f"Expected at least x and dose columns in {path}")
    arr = df.to_numpy(dtype=float)
    x, dose = arr[:,0], arr[:,1]
    unc = arr[:,2] if arr.shape[1] >= 3 else np.full_like(dose, np.nan)
    ok = np.isfinite(x) & np.isfinite(dose)
    order = np.argsort(x[ok])
    return pd.DataFrame({"x_mm": x[ok][order], "sim_raw": dose[ok][order], "sim_unc_raw": unc[ok][order]})

def locate_sim_file(energy, app_cm, depth):
    prefix = {"dmax":"latDmax", "r50":"latR50", "mid":"latMid"}[depth]
    matches = sorted(ROOT.glob(f"{prefix}_{energy}_{app_cm}cm_*.txt"))
    if not matches:
        raise FileNotFoundError(f"No simulation file for {energy} {app_cm} cm {depth}")
    return matches[0]

def normalize(y):
    y = np.asarray(y, dtype=float)
    m = np.nanmax(y)
    return 100.0*y/m if np.isfinite(m) and m != 0 else y*np.nan

def interp_profile(x, y, xnew):
    x, y, xnew = map(lambda a: np.asarray(a, dtype=float), (x, y, xnew))
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    order = np.argsort(x)
    x, y = x[order], y[order]
    ux, idx = np.unique(x, return_index=True)
    uy = y[idx]
    if PchipInterpolator and len(ux) >= 4:
        f = PchipInterpolator(ux, uy, extrapolate=False)
        return f(xnew)
    return np.interp(xnew, ux, uy, left=np.nan, right=np.nan)


In [ ]:
def measured_column_candidates(df, app_cm, depth):
    cols = list(df.columns)
    app_pat = re.compile(fr"(^|\D){app_cm}(\s*cm|cm|\D|$)", re.I)
    depth_terms = {"dmax":["dmax","d max"], "r50":["r50"], "mid":["mid","middle"]}[depth]
    scored = []
    for c in cols:
        s = str(c).lower()
        score = 0
        if app_pat.search(s): score += 4
        if any(t in s for t in depth_terms): score += 4
        if any(t in s for t in ["pos", "x", "distance", "mm"]): score += 1
        if any(t in s for t in ["dose", "signal", "%", "profile"]): score += 1
        scored.append((score, c))
    top = [c for score,c in sorted(scored, reverse=True) if score >= 4]
    numeric = [c for c in cols if pd.api.types.is_numeric_dtype(df[c])]
    pool = top or numeric
    # pick adjacent numeric pair with best labels; fallback to first two numeric columns
    pool = [c for c in pool if c in numeric]
    if len(pool) >= 2:
        return pool[0], pool[1]
    if len(numeric) >= 2:
        return numeric[0], numeric[1]
    raise ValueError(f"Could not identify measured x/dose columns for {app_cm} cm {depth}")

def read_measured_profile(energy, app_cm, depth):
    sheet = EXCEL_SHEETS[energy]
    df = pd.read_excel(EXCEL_FILE, sheet_name=sheet)
    xcol, ycol = measured_column_candidates(df, app_cm, depth)
    out = df[[xcol, ycol]].rename(columns={xcol:"x_mm", ycol:"meas_raw"}).dropna()
    out["x_mm"] = pd.to_numeric(out["x_mm"], errors="coerce")
    out["meas_raw"] = pd.to_numeric(out["meas_raw"], errors="coerce")
    return out.dropna().sort_values("x_mm"), str(xcol), str(ycol)


In [ ]:
def crossing(x, y, level, side):
    x, y = np.asarray(x), np.asarray(y)
    mask = x <= 0 if side == "left" else x >= 0
    xs, ys = x[mask], y[mask]
    if side == "left":
        order = np.argsort(xs)
    else:
        order = np.argsort(xs)
    xs, ys = xs[order], ys[order]
    v = ys - level
    idx = np.where(np.signbit(v[:-1]) != np.signbit(v[1:]))[0]
    if len(idx) == 0: return np.nan
    i = idx[-1] if side == "left" else idx[0]
    return xs[i] + (level-ys[i])*(xs[i+1]-xs[i])/(ys[i+1]-ys[i])

def metrics(x, y):
    y = normalize(y)
    l50, r50 = crossing(x,y,50,"left"), crossing(x,y,50,"right")
    l80, r80 = crossing(x,y,80,"left"), crossing(x,y,80,"right")
    l20, r20 = crossing(x,y,20,"left"), crossing(x,y,20,"right")
    field = r50 - l50
    left_pen = l80 - l20
    right_pen = r20 - r80
    xp = np.asarray(x)
    left = interp_profile(x, y, -np.abs(xp[xp>=0]))
    right = interp_profile(x, y, np.abs(xp[xp>=0]))
    sym = 100*np.nanmax(np.abs(left-right))/max(np.nanmax(y),1e-9)
    return {"left_50_mm":l50, "right_50_mm":r50, "fwhm_mm":field, "left_80_20_penumbra_mm":left_pen, "right_80_20_penumbra_mm":right_pen, "symmetry_pct":sym}

def gamma_1d(x_ref, d_ref, x_eval, d_eval, pct=2, dta=2, threshold=10):
    ref = np.asarray(d_ref); xr = np.asarray(x_ref); xe = np.asarray(x_eval); de = np.asarray(d_eval)
    out = np.full_like(ref, np.nan, dtype=float)
    high = ref >= threshold
    for i in np.where(high)[0]:
        dist2 = ((xe - xr[i])/dta)**2 + ((de - ref[i])/pct)**2
        out[i] = np.sqrt(np.nanmin(dist2))
    return out, 100*np.nanmean(out[high] <= 1.0)


In [ ]:
rows=[]
processed={}
for energy in ENERGIES:
    for app in APPLICATORS_CM:
        for depth in DEPTHS:
            meas, xcol, ycol = read_measured_profile(energy, app, depth)
            sim_path = locate_sim_file(energy, app, depth)
            sim = read_sim_profile(sim_path)
            sim["x_mm"] = apply_x_tuning(sim["x_mm"], energy, app, depth)
            meas["meas_pct"] = normalize(meas["meas_raw"])
            sim["sim_pct"] = normalize(sim["sim_raw"])
            sim["sim_unc_pct"] = 100*sim["sim_unc_raw"]/np.nanmax(sim["sim_raw"]) if np.isfinite(sim["sim_unc_raw"]).any() else np.nan
            xmin=max(meas.x_mm.min(), sim.x_mm.min()); xmax=min(meas.x_mm.max(), sim.x_mm.max())
            grid=np.arange(math.ceil(xmin*2)/2, math.floor(xmax*2)/2+0.001, 0.5)
            mgrid=interp_profile(meas.x_mm, meas.meas_pct, grid)
            sgrid=interp_profile(sim.x_mm, sim.sim_pct, grid)
            ugrid=interp_profile(sim.x_mm, sim.sim_unc_pct, grid)
            diff=sgrid-mgrid
            pctdiff=100*diff/np.where(np.abs(mgrid)>1e-9, mgrid, np.nan)
            gamma, pass_rate=gamma_1d(grid, mgrid, grid, sgrid, GAMMA_PERCENT, GAMMA_DTA_MM, LOW_DOSE_THRESHOLD_PCT)
            key=(energy,app,depth)
            processed[key] = pd.DataFrame({"x_mm":grid,"measured_pct":mgrid,"simulated_pct":sgrid,"sim_uncertainty_pct":ugrid,"dose_difference_pctpt":diff,"pointwise_percent_difference":pctdiff,"gamma_2pct_2mm":gamma})
            processed[key].to_csv(DATA/f"processed_{energy}_{app}cm_{depth}.csv", index=False)
            mm, sm = metrics(grid,mgrid), metrics(grid,sgrid)
            row={"energy":energy,"applicator_cm":app,"depth":depth,"measured_x_column":xcol,"measured_dose_column":ycol,"simulation_file":sim_path.name,"gamma_pass_rate_pct":pass_rate,"mean_abs_diff_pctpt":np.nanmean(np.abs(diff)),"max_abs_diff_pctpt":np.nanmax(np.abs(diff)),"mean_mc_uncertainty_pct":np.nanmean(ugrid)}
            for k in mm: row[f"measured_{k}"]=mm[k]; row[f"simulated_{k}"]=sm[k]; row[f"delta_{k}"]=sm[k]-mm[k]
            rows.append(row)
summary=pd.DataFrame(rows)
summary.to_csv(TABLE/"lateral_profile_validation_summary.csv", index=False)
summary


In [ ]:
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG/name, bbox_inches="tight"); plt.show()

for (energy,app,depth), df in processed.items():
    title=f"{energy} {app} cm {DEPTHS[depth]}"
    plt.figure(figsize=(6,4)); plt.plot(df.x_mm,df.measured_pct,label="Measured",lw=2); plt.plot(df.x_mm,df.simulated_pct,label="Monte Carlo",lw=1.8); plt.fill_between(df.x_mm, df.simulated_pct-df.sim_uncertainty_pct, df.simulated_pct+df.sim_uncertainty_pct, alpha=.2, label="MC uncertainty"); plt.xlabel("Lateral position (mm)"); plt.ylabel("Normalized dose (%)"); plt.title(title); plt.legend(); savefig(f"comparison_{energy}_{app}cm_{depth}.png")
    plt.figure(figsize=(6,3)); plt.axhline(0,color="k",lw=.8); plt.plot(df.x_mm, df.pointwise_percent_difference); plt.xlabel("Lateral position (mm)"); plt.ylabel("(MC-meas)/meas (%)"); plt.title(f"Pointwise percent difference: {title}"); savefig(f"percent_difference_{energy}_{app}cm_{depth}.png")
    plt.figure(figsize=(6,3)); plt.axhline(1,color="r",ls="--",label="Pass criterion"); plt.plot(df.x_mm, df.gamma_2pct_2mm); plt.ylim(bottom=0); plt.xlabel("Lateral position (mm)"); plt.ylabel("Gamma index"); plt.title(f"2%/2 mm gamma: {title}"); plt.legend(); savefig(f"gamma_{energy}_{app}cm_{depth}.png")


In [ ]:
# Main-paper multi-panel figures
for energy in ENERGIES:
    fig, axes = plt.subplots(3,3, figsize=(12,10), sharex=False)
    for r, app in enumerate(APPLICATORS_CM):
        for c, depth in enumerate(DEPTHS):
            df=processed[(energy,app,depth)]; ax=axes[r,c]
            ax.plot(df.x_mm,df.measured_pct,label="Measured",lw=1.8); ax.plot(df.x_mm,df.simulated_pct,label="MC",lw=1.5)
            ax.set_title(f"{app} cm {DEPTHS[depth]}"); ax.set_xlabel("x (mm)"); ax.set_ylabel("Dose (%)")
    axes[0,0].legend(); fig.suptitle(f"Lateral profile validation: {energy}"); savefig(f"main_paper_profiles_{energy}.png")

publication_cols=["energy","applicator_cm","depth","gamma_pass_rate_pct","delta_fwhm_mm","delta_left_80_20_penumbra_mm","delta_right_80_20_penumbra_mm","delta_symmetry_pct","mean_mc_uncertainty_pct"]
pub=summary[publication_cols].copy().round(3)
pub.to_csv(TABLE/"publication_ready_table.csv", index=False)
with open(TABLE/"publication_ready_table.md","w") as f:
    f.write(pub.to_markdown(index=False))
pub


In [ ]:
# Applicator output factors for 10, 5, and 2 cm, normalized to 10 cm for each energy/depth.
of_rows=[]
for energy in ENERGIES:
    for depth in DEPTHS:
        ref = processed[(energy,10,depth)].loc[processed[(energy,10,depth)].x_mm.abs().idxmin(), "simulated_pct"]
        for app in APPLICATORS_CM:
            df=processed[(energy,app,depth)]
            central=df.loc[df.x_mm.abs().idxmin(), "simulated_pct"]
            of_rows.append({"energy":energy,"depth":depth,"applicator_cm":app,"relative_output_factor_vs_10cm":central/ref})
output_factors=pd.DataFrame(of_rows)
output_factors.to_csv(TABLE/"applicator_output_factors.csv", index=False)
output_factors
